# 07 — Rejection Breakdown

**C4**: Stale-suppressed and threshold-rejected signals excluded from win-rate numerator;  
included in opportunity-count denominator.

**V5**: Shows capture rate = emitted / all evaluations.

Breaks down rejection reasons: `below_threshold`, `negative_edge`, `cooldown_active`, stale (no rejection_reason but decision=rejected).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
joined = pd.read_parquet('../data/raw/joined.parquet')

rejected = joined[joined['decision'] == 'rejected'].copy()
rejected['rejection_reason'] = rejected['rejection_reason'].fillna('unknown')

breakdown = rejected.groupby('rejection_reason').size().sort_values(ascending=False)
print('=== Rejection Breakdown ===')
print(breakdown.to_string())

In [ ]:
# C4: confirm rejections included in denominator but excluded from win-rate
total_evals  = len(joined)
emitted      = (joined['decision'] == 'emitted').sum()
total_reject = (joined['decision'] == 'rejected').sum()
capture_rate = emitted / total_evals if total_evals > 0 else float('nan')

print(f'\nOpportunity count (denominator): {total_evals}')
print(f'Emitted (numerator for capture): {emitted}')
print(f'Capture rate: {capture_rate:.1%}')
print(f'Rejected: {total_reject} ({total_reject/total_evals:.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rejection pie
axes[0].pie(breakdown.values, labels=breakdown.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Rejection Reasons')

# Funnel: evaluated → emitted → rejected
stages  = ['Evaluated', 'Emitted', 'Rejected']
counts  = [total_evals, emitted, total_reject]
colors  = ['steelblue', 'green', 'tomato']
axes[1].barh(stages, counts, color=colors)
axes[1].set_xlabel('Count')
axes[1].set_title('Signal Decision Split')
for i, v in enumerate(counts):
    axes[1].text(v + 0.5, i, str(v), va='center')

plt.tight_layout()
plt.savefig('../data/raw/rejections.png', dpi=120)
plt.show()